# TP — Détection d'objets avec Faster R-CNN (backbone ResNet-50) sur COCO

**Auteur :** Ismail En-niou — Master MQL, Faculté des Sciences de Tétouan

**Objectif.** Implémenter, entraîner et évaluer un détecteur d'objets **Faster R-CNN** dont le réseau d'extraction de caractéristiques est un **ResNet-50** (avec *Feature Pyramid Network*), sur le dataset **COCO** (*Common Objects in Context*, 80 catégories).

**Plan du notebook**

1. Configuration de l'environnement
2. Préparation des données COCO (téléchargement, format, exploration)
3. Construction du `Dataset` PyTorch et des `DataLoader`
4. Architecture du modèle Faster R-CNN + ResNet-50
5. Entraînement
6. Courbes d'apprentissage et interprétation
7. Évaluation COCO (mAP, AP par taille, AP par classe, courbe précision-rappel)
8. Exemples de détection sur des images
9. Comparaison avec le modèle de référence pré-entraîné sur COCO complet
10. Conclusion et pistes d'amélioration

> **Exécution (Kaggle, GPU).** Entraîner Faster R-CNN sur COCO 2017 complet (118 287 images, 12 époques) prend plusieurs jours sur un seul GPU. On entraîne donc sur un **sous-ensemble représentatif de COCO 2017** : **6 000 images de train2017** (80 classes) et on évalue sur **500 images de val2017** (jamais vues), pendant 6 époques — environ 2 h sur un GPU T4/P100.
>
> Paramètres modifiables dans `CONFIG` : `num_train_images`, `num_val_images`, `epochs`. Le mode `MODE = "coco128"` permet aussi de tester le notebook sur CPU avec 128 images.
>
> **Les interprétations des graphes sont calculées automatiquement à partir des résultats obtenus** (blocs « Interprétation ») : elles restent justes quel que soit le nombre d'images ou d'époques choisi.

## 1. Configuration de l'environnement

In [ ]:
%pip install -q pycocotools
import os, json, math, random, time, zipfile, urllib.request, contextlib, io, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet50_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.ops.misc import FrozenBatchNorm2d
from torchvision.transforms import functional as TF
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device = {DEVICE}")
if DEVICE.type == "cpu":
    print(f"Threads CPU utilisés : {torch.get_num_threads()}")

**Explication :** on importe les bibliothèques nécessaires :
- **PyTorch / torchvision** : tenseurs, réseaux de neurones et implémentation de référence de Faster R-CNN (`torchvision.models.detection`) ;
- **pycocotools** : l'API officielle COCO, utilisée pour lire les annotations et calculer les métriques standard (mAP) ;
- **matplotlib / pandas** : graphes et tableaux de résultats.

Le `DEVICE` est choisi automatiquement : GPU s'il existe, sinon CPU.

In [ ]:
MODE = "kaggle"   # "kaggle" : COCO 2017 attaché au notebook Kaggle (GPU)  |  "coco128" : 128 images (test CPU)
KAGGLE = MODE == "kaggle"

CONFIG = dict(
    data_root             = Path("/kaggle/working/data") if KAGGLE else Path("data"),
    weights_dir           = Path("weights"),
    output_dir            = Path("/kaggle/working/outputs") if KAGGLE else Path("outputs"),
    num_train_images      = 6000 if KAGGLE else None,   # sous-ensemble de COCO train2017
    num_val_images        = 500 if KAGGLE else 28,      # images de validation (val2017 sur Kaggle)
    num_train_eval_images = 500 if KAGGLE else None,    # images d'entraînement ré-évaluées pour mesurer l'écart train/val
    epochs                = 6 if KAGGLE else 15,
    batch_size            = 4 if KAGGLE else 2,
    num_workers           = 4 if KAGGLE else 2,
    lr                    = 0.01,     # taux d'apprentissage de base (SGD)
    momentum              = 0.9,
    weight_decay          = 1e-4,
    warmup_iters          = 500 if KAGGLE else 100,   # montée progressive du LR au début
    min_size              = 512 if KAGGLE else 448,   # petit côté de l'image après redimensionnement
    max_size              = 853 if KAGGLE else 640,
    trainable_layers      = 3,        # nb de blocs ResNet ré-entraînés (layer2, layer3, layer4)
    amp                   = torch.cuda.is_available(),   # précision mixte (accélère l'entraînement sur GPU)
    log_every             = 200,      # affichage de la perte toutes les N itérations
    vis_score_thresh      = 0.5,      # seuil de confiance pour l'affichage des détections
    run_reference         = True,     # section 9 : comparaison avec le modèle torchvision pré-entraîné sur COCO
    seed                  = 42,
)
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
if KAGGLE and DEVICE.type == "cpu":
    print("ATTENTION : aucun GPU détecté. Sur Kaggle : Settings -> Accelerator -> GPU T4 x2 (ou P100).")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed(CONFIG["seed"])

def interpretation(lines):
    # affiche une interprétation rédigée à partir des valeurs réellement obtenues
    print("Interprétation (calculée à partir des résultats) :")
    for l in lines:
        print("  •", l)

pd.Series({k: str(v) for k, v in CONFIG.items()}, name="valeur").to_frame()

**Explication :** toutes les hyper-paramètres sont centralisés dans `CONFIG` pour rendre l'expérience reproductible et facilement modifiable :
- **SGD avec momentum 0.9, LR 0.01, weight decay 1e-4** : réglages standard de Faster R-CNN (cf. *Detectron2* / *torchvision references*) ;
- **warm-up** : le LR augmente linéairement pendant les 100 premières itérations pour éviter une divergence au début (les têtes RPN/ROI sont initialisées aléatoirement) ;
- **taille d'image** : la configuration standard COCO est 800×1333 ; on la réduit à 512×853 pour diviser le temps de calcul par ~2,5 ;
- **sous-ensemble** : 6 000 images d'entraînement et 500 de validation, tirées aléatoirement (graine fixe) ;
- **AMP** (*Automatic Mixed Precision*) : sur GPU, une partie des calculs se fait en float16, ce qui accélère nettement l'entraînement sans perte de précision notable ;
- **`trainable_layers = 3`** : on gèle `conv1` et `layer1` du ResNet (caractéristiques bas niveau génériques) et on ré-entraîne `layer2`→`layer4`.
- la **graine aléatoire** fixe les tirages (split, augmentations) pour la reproductibilité.

## 2. Préparation des données COCO

### 2.1 Les classes COCO
COCO contient **80 catégories d'objets**. Leurs identifiants officiels vont de 1 à 90 avec des « trous » (11 identifiants inutilisés). Le modèle, lui, a besoin d'étiquettes **contiguës** : 0 = fond (*background*), 1…80 = objets. On construit donc les tables de correspondance.

In [ ]:
COCO_NAMES = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light',
 'fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant','bear',
 'zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
 'kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle','wine glass','cup',
 'fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut',
 'cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard',
 'cell phone','microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear',
 'hair drier','toothbrush']
COCO_CAT_IDS = [1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,19,20,21,22,23,24,25,27,28,31,32,33,34,35,36,37,38,
 39,40,41,42,43,44,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,67,70,72,73,74,75,76,77,78,79,
 80,81,82,84,85,86,87,88,89,90]
assert len(COCO_NAMES) == len(COCO_CAT_IDS) == 80

CAT2LABEL  = {cid: i + 1 for i, cid in enumerate(COCO_CAT_IDS)}   # id COCO -> étiquette modèle (1..80)
LABEL2CAT  = {v: k for k, v in CAT2LABEL.items()}                 # étiquette modèle -> id COCO
LABEL2NAME = {i + 1: n for i, n in enumerate(COCO_NAMES)}
NUM_CLASSES = len(COCO_NAMES) + 1                                 # 80 objets + fond
print("Nombre de classes du modèle (fond inclus) :", NUM_CLASSES)

**Explication :** `CAT2LABEL` convertit l'identifiant COCO (ex. 90 = *toothbrush*) en indice contigu (80) utilisé par la couche de classification ; `LABEL2CAT` fait l'inverse au moment de l'évaluation, car `COCOeval` attend les identifiants officiels. La classe 0 est réservée au **fond**, ce qui donne `NUM_CLASSES = 81` sorties pour la tête de classification.

### 2.2 Téléchargement des données et des poids pré-entraînés
- **Mode `coco128`** : archive officielle de 128 images COCO train2017 (Ultralytics) dont les labels sont au format YOLO → on les **convertit au format d'annotation COCO JSON** pour travailler exactement comme avec le dataset complet.
- **Mode `kaggle`** : COCO 2017 complet est déjà disponible sur Kaggle comme dataset ; il suffit de l'attacher au notebook (*Add Input*), aucun téléchargement de 19 Go n'est nécessaire.
- **Poids ResNet-50 ImageNet** : le backbone est initialisé avec des poids pré-entraînés sur ImageNet (*transfer learning*), ce qui est la pratique standard pour Faster R-CNN.

In [ ]:
COCO128_URL  = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"
RESNET50_URL = "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-rsb-weights/resnet50_a1_0-14fe96d1.pth"

def download(url, dest):
    dest = Path(dest)
    if dest.exists():
        print(f"[ok] déjà présent : {dest}")
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"[..] téléchargement : {url}")
    urllib.request.urlretrieve(url, dest)
    return dest

def unzip(zip_path, out_dir):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(out_dir)

root = CONFIG["data_root"]
root.mkdir(parents=True, exist_ok=True)

def find_coco2017(base):
    # parcourt les données attachées au notebook à la recherche des annotations et des dossiers d'images COCO 2017
    found = {}
    for dirpath, dirnames, filenames in os.walk(base, followlinks=True):
        for f in ("instances_train2017.json", "instances_val2017.json"):
            if f in filenames:
                found.setdefault(f, Path(dirpath) / f)
        for d in list(dirnames):
            if d in ("train2017", "val2017", "test2017"):
                dirnames.remove(d)                          # on ne parcourt pas les dossiers d'images
                p = Path(dirpath) / d
                if next(p.glob("*.jpg"), None) is None and (p / d).is_dir():
                    p = p / d                               # cas d'un dossier imbriqué « train2017/train2017 »
                if d != "test2017":
                    found.setdefault(d, p)
    return found

if MODE == "kaggle":
    BACKBONE_WEIGHTS = None     # poids ImageNet officiels de torchvision (téléchargés automatiquement)
    COCO_FOUND = find_coco2017(Path(os.environ.get("COCO_SEARCH_ROOT", "/kaggle/input")))
    for k, v in COCO_FOUND.items():
        print(f"{k:26s} -> {v}")
    manquants = {"train2017", "val2017", "instances_train2017.json", "instances_val2017.json"} - set(COCO_FOUND)
    assert not manquants, (f"Éléments COCO 2017 introuvables : {manquants}. Ajoutez le dataset COCO 2017 au "
                           "notebook : panneau de droite -> Add Input -> rechercher « COCO 2017 ».")
else:
    BACKBONE_WEIGHTS = download(RESNET50_URL, CONFIG["weights_dir"] / "resnet50_imagenet.pth")
    if not (root / "coco128").exists():
        unzip(download(COCO128_URL, root / "coco128.zip"), root)
    print("Images disponibles :", len(list((root / "coco128/images/train2017").glob("*.jpg"))))

**Explication :** sur Kaggle, le dataset COCO 2017 est attaché au notebook (dossier `/kaggle/input`, en lecture seule) : `find_coco2017` localise automatiquement les dossiers `train2017`, `val2017` et les fichiers `instances_*2017.json`, quelle que soit l'organisation du dataset choisi. Les poids du ResNet-50 pré-entraîné sur ImageNet-1k (classification, 1000 classes) sont fournis par torchvision ; on n'en gardera que la partie convolutionnelle. En mode `coco128`, `download` récupère les 128 images et les poids sans re-télécharger ce qui est déjà présent.

### 2.3 Construction des annotations au format COCO
Le format COCO JSON contient trois listes : `images` (id, nom de fichier, taille), `annotations` (boîte `[x, y, largeur, hauteur]` en pixels, catégorie, aire) et `categories`.
- **Kaggle** : on tire aléatoirement 6 000 images annotées de train2017 et 500 de val2017 et on écrit leurs annotations dans deux nouveaux fichiers COCO JSON (sous-ensembles du fichier officiel) ;
- **coco128** : on convertit les labels YOLO (centre et taille normalisés) vers ce format, avec un split 100 / 28.

In [ ]:
def yolo_to_coco_json(img_files, lbl_dir, out_json):
    images, annotations, ann_id = [], [], 1
    for f in img_files:
        img_id = int(f.stem)                      # ex : 000000000009.jpg -> 9 (identifiant COCO d'origine)
        W, H = Image.open(f).size
        images.append({"id": img_id, "file_name": f.name, "width": W, "height": H})
        lbl = lbl_dir / f"{f.stem}.txt"
        if not lbl.exists():
            continue
        for line in lbl.read_text().strip().splitlines():
            c, xc, yc, bw, bh = line.split()[:5]
            bw, bh = float(bw) * W, float(bh) * H
            x, y = float(xc) * W - bw / 2, float(yc) * H - bh / 2
            annotations.append({"id": ann_id, "image_id": img_id, "category_id": COCO_CAT_IDS[int(c)],
                                "bbox": [round(x, 2), round(y, 2), round(bw, 2), round(bh, 2)],
                                "area": bw * bh, "iscrowd": 0})
            ann_id += 1
    categories = [{"id": cid, "name": n, "supercategory": ""} for cid, n in zip(COCO_CAT_IDS, COCO_NAMES)]
    json.dump({"images": images, "annotations": annotations, "categories": categories}, open(out_json, "w"))
    return len(images), len(annotations)

if MODE == "coco128":
    img_dir = root / "coco128/images/train2017"
    files = sorted(img_dir.glob("*.jpg"))
    random.Random(CONFIG["seed"]).shuffle(files)
    val_files, train_files = files[:CONFIG["num_val_images"]], files[CONFIG["num_val_images"]:]
    TRAIN_IMG_DIR = VAL_IMG_DIR = img_dir
    TRAIN_ANN, VAL_ANN = root / "coco128_train.json", root / "coco128_val.json"
    for name, fl, out in [("train", train_files, TRAIN_ANN), ("val", val_files, VAL_ANN)]:
        n_img, n_ann = yolo_to_coco_json(fl, root / "coco128/labels/train2017", out)
        print(f"{name:5s}: {n_img:3d} images, {n_ann:4d} objets annotés -> {out}")
else:
    def make_subset(ann_file, n, out_json, seed):
        # garde n images (ayant au moins un objet annoté) et toutes leurs annotations, au format COCO
        data = json.load(open(ann_file))
        annotated = {a["image_id"] for a in data["annotations"] if not a.get("iscrowd", 0)}
        imgs = sorted((im for im in data["images"] if im["id"] in annotated), key=lambda im: im["id"])
        random.Random(seed).shuffle(imgs)
        imgs = imgs[:n]
        keep = {im["id"] for im in imgs}
        anns = [a for a in data["annotations"] if a["image_id"] in keep]
        json.dump({"images": imgs, "annotations": anns, "categories": data["categories"]}, open(out_json, "w"))
        return len(data["images"]), len(imgs), len(anns)

    TRAIN_IMG_DIR, VAL_IMG_DIR = COCO_FOUND["train2017"], COCO_FOUND["val2017"]
    TRAIN_ANN, VAL_ANN = root / "coco_train_subset.json", root / "coco_val_subset.json"
    for name, src_file, n, out in [("train", COCO_FOUND["instances_train2017.json"], CONFIG["num_train_images"], TRAIN_ANN),
                                   ("val", COCO_FOUND["instances_val2017.json"], CONFIG["num_val_images"], VAL_ANN)]:
        total, n_img, n_ann = make_subset(src_file, n, out, CONFIG["seed"])
        print(f"{name:5s}: {n_img:5d} images retenues sur {total} | {n_ann:6d} objets annotés -> {out}")

**Explication :** `make_subset` lit le fichier d'annotations officiel, ne garde que les images contenant au moins un objet, en tire `n` au hasard (graine fixe → sous-ensemble reproductible) et réécrit un fichier COCO JSON avec ces images et leurs annotations. Les images de **validation proviennent de val2017** : elles n'ont jamais été vues à l'entraînement. À partir de là, **tout le reste du notebook ne manipule que des fichiers COCO JSON standard** (lecture avec l'API `COCO`, évaluation avec `COCOeval`).

### 2.4 Exploration des données

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):          # masque les messages de chargement de pycocotools
    coco_train, coco_val = COCO(str(TRAIN_ANN)), COCO(str(VAL_ANN))

def ann_dataframe(coco, split):
    rows = []
    for a in coco.dataset["annotations"]:
        img = coco.imgs[a["image_id"]]
        rows.append({"split": split, "classe": coco.cats[a["category_id"]]["name"],
                     "w": a["bbox"][2], "h": a["bbox"][3], "aire": a["bbox"][2] * a["bbox"][3],
                     "aire_rel": a["bbox"][2] * a["bbox"][3] / (img["width"] * img["height"])})
    return pd.DataFrame(rows)

df = pd.concat([ann_dataframe(coco_train, "train"), ann_dataframe(coco_val, "val")])
df["taille"] = pd.cut(df["aire"], [0, 32**2, 96**2, np.inf], labels=["small (<32²)", "medium", "large (>96²)"])

summary = pd.DataFrame({
    "images": [len(coco_train.imgs), len(coco_val.imgs)],
    "objets": [len(coco_train.anns), len(coco_val.anns)],
    "classes présentes": [df[df.split == s].classe.nunique() for s in ["train", "val"]],
}, index=["train", "val"])
summary["objets / image"] = (summary["objets"] / summary["images"]).round(2)
summary

**Explication :** on charge les deux fichiers d'annotations avec l'API `COCO` et on les transforme en `DataFrame` (une ligne par objet) pour calculer des statistiques : nombre d'images, d'objets, de classes et la densité d'objets par image. On classe aussi chaque objet selon les **catégories de taille officielles COCO** : *small* (< 32² px), *medium* (32²–96²), *large* (> 96²), qui servent aussi à l'évaluation (AP_S, AP_M, AP_L).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw={"width_ratios": [2.2, 1, 1]})

top = df.classe.value_counts().head(20)
counts = df.groupby(["classe", "split"]).size().unstack(fill_value=0).loc[top.index]
counts.plot.barh(stacked=True, ax=axes[0], color=["#4C72B0", "#DD8452"])
axes[0].invert_yaxis(); axes[0].set_title("Top 20 des classes (nombre d'objets)"); axes[0].set_xlabel("objets")

df.groupby(["taille", "split"]).size().unstack().plot.bar(ax=axes[1], color=["#4C72B0", "#DD8452"], rot=0)
axes[1].set_title("Répartition par taille (catégories COCO)"); axes[1].set_xlabel("")

per_img = pd.Series([len(coco_train.getAnnIds(imgIds=i)) for i in coco_train.imgs] +
                    [len(coco_val.getAnnIds(imgIds=i)) for i in coco_val.imgs])
axes[2].hist(per_img, bins=25, color="#55A868")
axes[2].set_title("Nombre d'objets par image"); axes[2].set_xlabel("objets"); axes[2].set_ylabel("images")
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "exploration.png", bbox_inches="tight"); plt.show()

print("Part de 'person' :", f"{(df.classe == 'person').mean():.1%}")
print(df.groupby("split").taille.value_counts(normalize=True).unstack().round(3))

**Explication :** trois graphes décrivent le dataset :
1. la **fréquence des classes** (empilée train/val) — elle révèle le déséquilibre ;
2. la **répartition par taille** des objets ;
3. la **densité** (nombre d'objets par image).

**Interprétation :** (voir plus bas, section *Analyse des données*).

#### Analyse des données
COCO est volontairement difficile : beaucoup d'objets par image, une classe *person* ultra-dominante, des classes rares et une forte proportion de petits objets. Ces caractéristiques expliqueront plus loin une grande partie des performances par classe et par taille.

In [ ]:
n_cls = df.classe.value_counts()
p_small = (df.taille == "small (<32²)").mean()
interpretation([
    f"La classe « {n_cls.index[0]} » représente {n_cls.iloc[0] / len(df):.0%} des objets, alors que {(n_cls < 50).sum()} classes "
    f"ont moins de 50 exemples : le dataset est très déséquilibré, le modèle verra beaucoup plus de personnes que d'objets rares.",
    f"{p_small:.0%} des objets sont petits (< 32×32 px) : c'est la catégorie la plus difficile pour un détecteur, "
    "car ils ne couvrent que quelques cellules des cartes de caractéristiques.",
    f"En moyenne {per_img.mean():.1f} objets par image (jusqu'à {per_img.max()}) : scènes denses, avec occlusions et chevauchements.",
])

In [ ]:
def draw_boxes(ax, img, boxes, labels, scores=None, color="lime", thresh=0.0):
    # boxes au format [x1, y1, x2, y2]
    ax.imshow(img); ax.axis("off")
    for i, (b, l) in enumerate(zip(boxes, labels)):
        s = None if scores is None else scores[i]
        if s is not None and s < thresh:
            continue
        x1, y1, x2, y2 = b
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=2))
        txt = LABEL2NAME[int(l)] + ("" if s is None else f" {s:.2f}")
        ax.text(x1, y1 - 2, txt, fontsize=7.5, color="black", bbox=dict(facecolor=color, alpha=0.75, pad=1, lw=0))

def gt_boxes(coco, img_id):
    anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=False))
    boxes = [[a["bbox"][0], a["bbox"][1], a["bbox"][0] + a["bbox"][2], a["bbox"][1] + a["bbox"][3]] for a in anns]
    return boxes, [CAT2LABEL[a["category_id"]] for a in anns]

sample_ids = random.Random(0).sample(sorted(coco_train.imgs), 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, img_id in zip(axes.ravel(), sample_ids):
    img = Image.open(TRAIN_IMG_DIR / coco_train.imgs[img_id]["file_name"])
    draw_boxes(ax, img, *gt_boxes(coco_train, img_id))
    ax.set_title(f"image {img_id}  ({len(gt_boxes(coco_train, img_id)[0])} objets)", fontsize=9)
plt.suptitle("Exemples d'images d'entraînement avec leurs annotations (vérité terrain)", fontsize=12)
plt.tight_layout(); plt.show()

**Explication :** `draw_boxes` dessine des boîtes `[x1, y1, x2, y2]` avec leur étiquette (et le score de confiance pour les prédictions). On l'utilise ici pour vérifier visuellement que la conversion des annotations est correcte : les boîtes vertes entourent bien les objets. On retrouve la difficulté propre à COCO : scènes encombrées, objets partiellement cachés, très petits objets et forte variabilité d'échelle.

## 3. Dataset PyTorch et DataLoaders
Faster R-CNN de torchvision attend, pour chaque image, un tenseur `[3, H, W]` à valeurs dans [0, 1] et une **cible** (`target`) : un dictionnaire contenant `boxes` (tenseur `[N, 4]` au format `x1, y1, x2, y2`) et `labels` (tenseur `[N]` d'entiers).

In [ ]:
class CocoDetDataset(Dataset):
    # Dataset COCO générique : lit les images + annotations JSON et produit (image, target)
    def __init__(self, img_dir, coco, train=False):
        self.img_dir, self.coco, self.train = Path(img_dir), coco, train
        ids = sorted(coco.imgs)
        if train:   # à l'entraînement on ignore les images sans objet annoté
            ids = [i for i in ids if len(coco.getAnnIds(imgIds=i, iscrowd=False)) > 0]
        self.ids = ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.imgs[img_id]
        img = TF.to_tensor(Image.open(self.img_dir / info["file_name"]).convert("RGB"))

        boxes, labels = [], []
        for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id, iscrowd=False)):
            x, y, w, h = a["bbox"]
            if w >= 1 and h >= 1:                                  # on écarte les boîtes dégénérées
                boxes.append([x, y, x + w, y + h])                 # COCO [x,y,w,h] -> [x1,y1,x2,y2]
                labels.append(CAT2LABEL[a["category_id"]])
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if self.train and random.random() < 0.5:                   # augmentation : flip horizontal
            img = img.flip(-1)
            W = img.shape[-1]
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]

        return img, {"boxes": boxes, "labels": labels, "image_id": img_id}

def collate_fn(batch):
    # les images ont des tailles différentes : on garde des listes au lieu d'empiler en un seul tenseur
    return tuple(zip(*batch))

train_ds = CocoDetDataset(TRAIN_IMG_DIR, coco_train, train=True)
val_ds   = CocoDetDataset(VAL_IMG_DIR, coco_val, train=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate_fn,
                          num_workers=CONFIG["num_workers"], pin_memory=DEVICE.type == "cuda")
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=CONFIG["num_workers"])

img, tgt = train_ds[0]
print(f"train : {len(train_ds)} images ({len(train_loader)} batchs) | val : {len(val_ds)} images")
print("image :", tuple(img.shape), img.dtype, f"[{img.min():.2f}, {img.max():.2f}]")
print("target:", {k: (tuple(v.shape) if torch.is_tensor(v) else v) for k, v in tgt.items()})

**Explication :**
- `__getitem__` charge l'image, la convertit en tenseur et transforme chaque annotation COCO `[x, y, w, h]` en `[x1, y1, x2, y2]` ; l'identifiant COCO est converti en étiquette contiguë via `CAT2LABEL` ;
- **augmentation de données** : un *flip* horizontal aléatoire (p = 0.5) ; il faut alors aussi retourner les boîtes (`x1' = W − x2`, `x2' = W − x1`) ;
- la **normalisation** (moyenne/écart-type ImageNet) et le **redimensionnement** ne sont pas faits ici : ils sont réalisés à l'intérieur du modèle par `GeneralizedRCNNTransform` ;
- `collate_fn` : les images n'ont pas toutes la même taille, on renvoie donc des tuples d'images et de cibles ; le modèle se charge de les regrouper (*padding*) en un batch.

## 4. Architecture du modèle : Faster R-CNN + ResNet-50-FPN

Faster R-CNN (Ren et al., 2015) est un détecteur **à deux étapes** :

| Étape | Composant | Rôle |
|---|---|---|
| 0 | **Backbone ResNet-50** | extrait des cartes de caractéristiques (C2…C5, strides 4→32) à partir de l'image |
| 0 bis | **FPN** (*Feature Pyramid Network*) | combine les niveaux (top-down + connexions latérales) → pyramide P2…P6 de 256 canaux, utile pour détecter des objets de toutes tailles |
| 1 | **RPN** (*Region Proposal Network*) | glisse sur chaque niveau de la pyramide, prédit pour 3 *anchors* par position un score « objet / fond » et un ajustement de boîte → ~2000 **propositions** (après NMS) |
| 2 | **RoI Align** | découpe pour chaque proposition une carte de caractéristiques de taille fixe 7×7 |
| 2 bis | **Tête Box (Fast R-CNN)** | deux couches FC (1024) puis : **classification** en 81 classes et **régression** des boîtes par classe |
| — | **Post-traitement** | seuil de score, NMS par classe (IoU 0.5), 100 détections max par image |

**Fonction de perte** (somme de 4 termes) :
- `loss_objectness` : entropie croisée binaire du RPN (objet vs fond) ;
- `loss_rpn_box_reg` : *Smooth L1* sur le raffinement des anchors ;
- `loss_classifier` : entropie croisée sur les 81 classes (tête ROI) ;
- `loss_box_reg` : *Smooth L1* sur le raffinement final des boîtes.

In [ ]:
def build_faster_rcnn(num_classes, backbone_weights, trainable_layers=3, min_size=800, max_size=1333):
    # 1) Backbone ResNet-50 + FPN (BatchNorm gelées : batch trop petit pour estimer des statistiques fiables)
    if backbone_weights is None:
        # 2a) poids ImageNet officiels de torchvision (la couche 'fc' de classification est retirée)
        backbone = resnet_fpn_backbone(backbone_name="resnet50", weights=ResNet50_Weights.IMAGENET1K_V1,
                                       norm_layer=FrozenBatchNorm2d, trainable_layers=trainable_layers)
        print("Poids ImageNet chargés dans ResNet-50 (torchvision ResNet50_Weights.IMAGENET1K_V1)")
    else:
        backbone = resnet_fpn_backbone(backbone_name="resnet50", weights=None,
                                       norm_layer=FrozenBatchNorm2d, trainable_layers=trainable_layers)
        # 2b) chargement d'un fichier de poids ImageNet (la couche 'fc' de classification est ignorée)
        state = torch.load(backbone_weights, map_location="cpu")
        state = {k: v for k, v in state.items() if not k.startswith("fc.")}
        missing, unexpected = backbone.body.load_state_dict(state, strict=False)
        print(f"Poids ImageNet chargés dans ResNet-50 | clés manquantes : {len(missing)} | inattendues : {len(unexpected)}")
    # 3) Assemblage Faster R-CNN : RPN + RoI Align + tête de classification / régression
    #    box_score_thresh=0.01 : on garde aussi les détections peu confiantes pour le calcul de la mAP
    model = FasterRCNN(backbone, num_classes=num_classes, min_size=min_size, max_size=max_size,
                       box_score_thresh=0.01, box_detections_per_img=100)
    return model

model = build_faster_rcnn(NUM_CLASSES, BACKBONE_WEIGHTS, CONFIG["trainable_layers"],
                          CONFIG["min_size"], CONFIG["max_size"]).to(DEVICE)

def count(params):
    return sum(p.numel() for p in params)
parts = {"Backbone ResNet-50": model.backbone.body, "FPN": model.backbone.fpn,
         "RPN": model.rpn, "Tête RoI (box head + predictor)": model.roi_heads}
param_table = pd.DataFrame({
    "paramètres (M)": [count(m.parameters()) / 1e6 for m in parts.values()],
    "entraînables (M)": [count(p for p in m.parameters() if p.requires_grad) / 1e6 for m in parts.values()],
}, index=parts.keys()).round(2)
param_table.loc["TOTAL"] = param_table.sum()
param_table

**Explication :**
- `resnet_fpn_backbone` crée un ResNet-50 tronqué (sans la couche `fc` finale) et lui ajoute le FPN. Les **BatchNorm sont gelées** (`FrozenBatchNorm2d`) : avec 2 images par batch, les statistiques de batch seraient trop bruitées ;
- le ResNet-50 est initialisé avec les **poids ImageNet** (*transfer learning*) : il sait déjà extraire des contours, textures et formes ; seuls le FPN, le RPN et la tête de détection partent de zéro ;
- `FasterRCNN(...)` ajoute automatiquement le générateur d'*anchors* (tailles 32→512, ratios 0.5/1/2), le **RPN**, le **MultiScaleRoIAlign** et la **tête de prédiction** à 81 classes ;
- le tableau montre la répartition des ~41 M de paramètres : le backbone ResNet-50 en représente plus de la moitié, mais `conv1`/`layer1` sont gelés ; la tête RoI (deux couches FC de 1024 neurones) est la plus grosse partie entièrement apprise depuis zéro.

## 5. Entraînement

On définit d'abord une fonction d'**évaluation COCO** (réutilisée à chaque époque), puis la boucle d'entraînement.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = {}
    for imgs, tgts in loader:
        outs = model([im.to(DEVICE) for im in imgs])
        for out, t in zip(outs, tgts):
            preds[t["image_id"]] = {k: v.cpu() for k, v in out.items()}
    return preds

def coco_evaluate(coco_gt, preds, verbose=False):
    # convertit les prédictions au format résultat COCO puis calcule les 12 métriques standard
    results = []
    for img_id, p in preds.items():
        b = p["boxes"].clone()
        b[:, 2:] -= b[:, :2]                                       # [x1,y1,x2,y2] -> [x,y,w,h]
        for box, s, l in zip(b.tolist(), p["scores"].tolist(), p["labels"].tolist()):
            results.append({"image_id": img_id, "category_id": LABEL2CAT[l], "bbox": box, "score": s})
    if not results:
        return np.zeros(12), None
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        coco_dt = coco_gt.loadRes(results)
        E = COCOeval(coco_gt, coco_dt, "bbox")
        E.params.imgIds = sorted(preds.keys())
        E.evaluate(); E.accumulate(); E.summarize()
    if verbose:
        print(out.getvalue().split("DONE")[-1].strip())
    return E.stats, E

METRIC_NAMES = ["AP@[.5:.95]", "AP@.5", "AP@.75", "AP small", "AP medium", "AP large",
                "AR@1", "AR@10", "AR@100", "AR small", "AR medium", "AR large"]

**Explication :**
- `predict` passe le modèle en mode `eval()` (en inférence, Faster R-CNN renvoie des boîtes, scores et labels au lieu des pertes) ;
- `coco_evaluate` convertit les boîtes au format COCO `[x, y, w, h]`, réassocie l'identifiant de catégorie officiel puis appelle `COCOeval`, qui calcule les **12 métriques standard** :
  - **AP@[.5:.95]** (*mAP* COCO, métrique principale) : précision moyenne, moyennée sur 10 seuils d'IoU de 0.50 à 0.95 et sur les 80 classes ;
  - **AP@.5** (métrique « PASCAL VOC ») et **AP@.75** (localisation stricte) ;
  - **AP small / medium / large** selon la taille des objets ;
  - **AR@k** : rappel moyen avec au plus *k* détections par image.

In [ ]:
RUN_TRAINING = True          # passer à False pour recharger le dernier checkpoint sans ré-entraîner
CKPT_PATH = CONFIG["output_dir"] / "faster_rcnn_r50_fpn.pth"
HIST_PATH = CONFIG["output_dir"] / "history.json"

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=CONFIG["lr"], momentum=CONFIG["momentum"], weight_decay=CONFIG["weight_decay"])
total_iters = CONFIG["epochs"] * len(train_loader)

def lr_factor(it):
    # warm-up linéaire puis décroissance cosinus jusqu'à 1 % du LR initial
    if it < CONFIG["warmup_iters"]:
        return 0.001 + (1 - 0.001) * it / CONFIG["warmup_iters"]
    progress = (it - CONFIG["warmup_iters"]) / max(1, total_iters - CONFIG["warmup_iters"])
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=CONFIG["amp"])    # mise à l'échelle des gradients (précision mixte)
print(f"{len(train_loader)} itérations par époque, {total_iters} au total")

def train_one_epoch(epoch, history):
    model.train()
    for imgs, tgts in train_loader:
        imgs = [im.to(DEVICE) for im in imgs]
        tgts = [{k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in t.items()} for t in tgts]
        with torch.autocast(device_type=DEVICE.type, enabled=CONFIG["amp"]):   # précision mixte sur GPU
            loss_dict = model(imgs, tgts)              # mode train : le modèle renvoie les 4 pertes
            loss = sum(loss_dict.values())
        if not math.isfinite(loss.item()):
            print("perte non finie, batch ignoré"); continue
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, max_norm=10.0)   # stabilise le début de l'apprentissage
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        history["iter"].append(len(history["iter"]) + 1)
        history["epoch"].append(epoch)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["loss"].append(loss.item())
        for k, v in loss_dict.items():
            history.setdefault(k, []).append(v.item())
        if len(history["iter"]) % CONFIG["log_every"] == 0:
            print(f"   it {len(history['iter']):5d}/{total_iters} | perte {np.mean(history['loss'][-CONFIG['log_every']:]):.3f}"
                  f" | lr {history['lr'][-1]:.5f}", flush=True)

if RUN_TRAINING:
    history, epoch_log = {"iter": [], "epoch": [], "lr": [], "loss": []}, []
    best_map, t0 = -1, time.time()
    for epoch in range(1, CONFIG["epochs"] + 1):
        te = time.time()
        train_one_epoch(epoch, history)
        stats, _ = coco_evaluate(coco_val, predict(model, val_loader))
        ep_losses = [l for l, e in zip(history["loss"], history["epoch"]) if e == epoch]
        epoch_log.append({"epoch": epoch, "train_loss": float(np.mean(ep_losses)),
                          "val_mAP": float(stats[0]), "val_AP50": float(stats[1]), "val_AP75": float(stats[2]),
                          "val_AR100": float(stats[8]), "lr": history["lr"][-1], "durée_s": time.time() - te})
        if stats[0] >= best_map:
            best_map = stats[0]
            torch.save(model.state_dict(), CKPT_PATH)
        print(f"Époque {epoch:2d}/{CONFIG['epochs']} | perte {epoch_log[-1]['train_loss']:.3f} | "
              f"val mAP {stats[0]:.4f} | AP50 {stats[1]:.4f} | {time.time() - te:.0f}s")
        json.dump({"history": history, "epoch_log": epoch_log}, open(HIST_PATH, "w"))
    print(f"\nEntraînement terminé en {(time.time() - t0) / 60:.1f} min — meilleur val mAP = {best_map:.4f}")
else:
    saved = json.load(open(HIST_PATH)); history, epoch_log = saved["history"], saved["epoch_log"]

model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))   # on garde le meilleur modèle (val mAP)
epoch_df = pd.DataFrame(epoch_log).set_index("epoch")
epoch_df.round(4)

**Explication de la boucle d'entraînement :**
1. **Optimiseur SGD** sur les seuls paramètres entraînables (`requires_grad=True`) ;
2. **Planification du LR** : warm-up linéaire (100 itérations, de 0.1 % à 100 % du LR) puis **décroissance cosinus** ; cela permet de grands pas au début et un réglage fin à la fin ;
3. pour chaque batch : passage avant en mode `train()` (en **précision mixte** sur GPU) → le modèle renvoie directement le dictionnaire des **4 pertes**, qu'on additionne ; rétro-propagation (avec `GradScaler` pour éviter les sous-dépassements en float16) ; **gradient clipping** (norme ≤ 10) pour éviter les explosions de gradient au démarrage ; mise à jour des poids ;
4. à la fin de chaque époque : **évaluation COCO sur la validation** et sauvegarde du **meilleur modèle** selon la val mAP (*early-stopping* implicite) ;
5. tout l'historique (pertes par itération, métriques par époque) est sauvegardé en JSON pour tracer les courbes et pouvoir recharger les résultats sans ré-entraîner (`RUN_TRAINING = False`).

## 6. Courbes d'apprentissage

In [ ]:
h = pd.DataFrame(history)
win = max(25, len(h) // 60)
smooth = lambda s: s.rolling(win, min_periods=1).mean()
loss_keys = ["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
axes[0].plot(h.iter, h.loss, alpha=0.25, color="gray", label="par itération")
axes[0].plot(h.iter, smooth(h.loss), color="black", lw=2, label=f"moyenne glissante ({win})")
axes[0].set_title("Perte totale d'entraînement"); axes[0].set_xlabel("itération"); axes[0].legend()

for k in loss_keys:
    axes[1].plot(h.iter, smooth(h[k]), lw=2, label=k)
axes[1].set_title("Décomposition de la perte (moyenne glissante)"); axes[1].set_xlabel("itération"); axes[1].legend()

axes[2].plot(h.iter, h.lr, color="#C44E52", lw=2)
axes[2].set_title("Taux d'apprentissage (warm-up + cosinus)"); axes[2].set_xlabel("itération")
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "loss_curves.png", bbox_inches="tight"); plt.show()

comp = h.groupby("epoch")[loss_keys + ["loss"]].mean()
print("Perte moyenne par composante — première vs dernière époque :")
pd.DataFrame({"époque 1": comp.iloc[0], f"époque {comp.index[-1]}": comp.iloc[-1],
              "réduction": (1 - comp.iloc[-1] / comp.iloc[0]).map("{:.0%}".format)}).round(3)

**Explication :** on trace (1) la perte totale par itération et sa moyenne glissante, (2) l'évolution de chacune des 4 composantes, (3) le taux d'apprentissage. Le tableau compare la perte moyenne de la première et de la dernière époque.

**Interprétation :**

Points à observer : la perte doit chuter fortement au début (le RPN et la tête apprennent à séparer objet / fond), puis décroître lentement ; la décroissance cosinus du LR lisse la fin de la courbe. Les pertes du RPN (`objectness`, `rpn_box_reg`) convergent vite car la tâche « objet ou fond » est simple ; la perte de **classification** à 81 classes reste généralement la plus élevée.

In [ ]:
first, last = comp.iloc[0], comp.iloc[-1]
red = 1 - last / first
interpretation([
    f"La perte totale passe de {first['loss']:.3f} (époque 1) à {last['loss']:.3f} (dernière époque), soit {red['loss']:.0%} de réduction : "
    "le modèle apprend effectivement.",
    f"Composante qui diminue le plus : {red[loss_keys].idxmax()} (−{red[loss_keys].max():.0%}) ; "
    f"celle qui diminue le moins : {red[loss_keys].idxmin()} (−{red[loss_keys].min():.0%}).",
    f"En fin d'entraînement, la plus grande composante est {last[loss_keys].idxmax()} ({last[loss_keys].max():.3f}) : "
    + ("reconnaître la bonne classe parmi 80 reste la partie la plus difficile."
       if last[loss_keys].idxmax() == "loss_classifier" else "c'est la sous-tâche sur laquelle le modèle progresse le moins."),
    "Les fortes oscillations d'une itération à l'autre sont normales : chaque batch ne contient que quelques images, "
    "de difficultés très différentes.",
])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
axes[0].plot(epoch_df.index, epoch_df.val_mAP, "o-", lw=2, label="mAP @[.5:.95]")
axes[0].plot(epoch_df.index, epoch_df.val_AP50, "s-", lw=2, label="AP @.5")
axes[0].plot(epoch_df.index, epoch_df.val_AP75, "^-", lw=2, label="AP @.75")
axes[0].plot(epoch_df.index, epoch_df.val_AR100, "d--", lw=1.5, label="AR @100")
best = epoch_df.val_mAP.idxmax()
axes[0].axvline(best, color="gray", ls=":", label=f"meilleure époque ({best})")
axes[0].set_title("Métriques COCO sur la validation, par époque"); axes[0].set_xlabel("époque"); axes[0].legend()

ax2 = axes[1]; ax3 = ax2.twinx()
ax2.plot(epoch_df.index, epoch_df.train_loss, "o-", color="black", lw=2, label="perte d'entraînement")
ax3.plot(epoch_df.index, epoch_df.val_AP50, "s-", color="#DD8452", lw=2, label="val AP@.5")
ax2.set_xlabel("époque"); ax2.set_ylabel("perte"); ax3.set_ylabel("AP@.5"); ax3.grid(False)
ax2.set_title("Perte d'entraînement vs performance en validation")
ax2.legend(loc="upper center"); ax3.legend(loc="center right")
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "val_metrics.png", bbox_inches="tight"); plt.show()

**Explication :** le graphe de gauche suit les métriques de validation au fil des époques ; celui de droite superpose la perte d'entraînement et l'AP@.5 de validation pour détecter un éventuel **sur-apprentissage** (perte qui continue de baisser alors que la validation stagne).

**Interprétation :**

Points à observer : la mAP doit augmenter au fil des époques ; si elle plafonne alors que la perte d'entraînement continue de baisser, le modèle commence à sur-apprendre. L'écart entre AP@.5 et AP@.75 mesure la qualité de la **localisation** des boîtes.

In [ ]:
best = int(epoch_df.val_mAP.idxmax()); last_e = int(epoch_df.index[-1])
lines = [
    f"Meilleure mAP de validation : {epoch_df.val_mAP.max():.4f} à l'époque {best} (AP@.5 = {epoch_df.loc[best, 'val_AP50']:.4f}).",
    f"La mAP passe de {epoch_df.val_mAP.iloc[0]:.4f} après la 1re époque à {epoch_df.val_mAP.iloc[-1]:.4f} après la dernière.",
]
if best < last_e:
    lines.append(f"La mAP n'augmente plus après l'époque {best} alors que la perte d'entraînement continue de baisser : "
                 "le modèle atteint un plateau / commence à sur-apprendre ; le meilleur checkpoint (époque "
                 f"{best}) est celui qui est conservé.")
else:
    lines.append("La mAP est maximale à la dernière époque : le modèle n'a pas encore convergé ; davantage d'époques "
                 "ou d'images amélioreraient encore les résultats.")
lines.append(f"AP@.5 = {epoch_df.val_AP50.iloc[-1]:.3f} contre AP@.75 = {epoch_df.val_AP75.iloc[-1]:.3f} : "
             "trouver l'objet est plus facile que l'encadrer très précisément.")
interpretation(lines)

## 7. Évaluation détaillée du meilleur modèle

### 7.1 Métriques COCO standard — validation vs entraînement

In [ ]:
t0 = time.time()
val_preds = predict(model, val_loader)
fps = len(val_ds) / (time.time() - t0)
print("=== Validation ===")
val_stats, E_val = coco_evaluate(coco_val, val_preds, verbose=True)

train_eval_ds = CocoDetDataset(TRAIN_IMG_DIR, coco_train, train=False)
if CONFIG["num_train_eval_images"] and len(train_eval_ds.ids) > CONFIG["num_train_eval_images"]:
    shown = set(sample_ids)          # on garde les images affichées en 2.4 pour la section 8.2
    others = [i for i in train_eval_ds.ids if i not in shown]
    train_eval_ds.ids = list(sample_ids) + random.Random(0).sample(others, CONFIG["num_train_eval_images"] - len(shown))
train_eval_loader = DataLoader(train_eval_ds, batch_size=4, collate_fn=collate_fn, num_workers=CONFIG["num_workers"])
train_preds = predict(model, train_eval_loader)
train_stats, E_train = coco_evaluate(coco_train, train_preds)

metrics_df = pd.DataFrame({"train": train_stats, "validation": val_stats}, index=METRIC_NAMES).round(4)
print(f"\nVitesse d'inférence ({DEVICE.type}) : {fps:.2f} images/s ({1000 / fps:.0f} ms/image)")
metrics_df

**Explication :** on recharge le **meilleur checkpoint**, on prédit sur la validation et on affiche le rapport officiel de `COCOeval`. On évalue aussi le modèle sur les images **d'entraînement** : l'écart train/validation mesure la **capacité de généralisation**. On mesure enfin la vitesse d'inférence.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
sel = ["AP@[.5:.95]", "AP@.5", "AP@.75", "AP small", "AP medium", "AP large", "AR@100"]
metrics_df.loc[sel].clip(lower=0).plot.bar(ax=axes[0], color=["#4C72B0", "#DD8452"], rot=30)
axes[0].set_title("Métriques COCO : entraînement vs validation"); axes[0].set_ylabel("score")

# Courbe précision-rappel : E.eval["precision"] a la forme [T(IoU), R(rappel), K(classe), A(aire), M(maxDets)]
recall = np.linspace(0, 1, 101)
for E, name, ls in [(E_val, "validation", "-"), (E_train, "train", "--")]:
    for t, iou in [(0, 0.5), (5, 0.75)]:
        p = E.eval["precision"][t, :, :, 0, 2]                   # aire = all, maxDets = 100
        valid = (p > -1).all(axis=0)                             # classes présentes dans ce split
        axes[1].plot(recall, p[:, valid].mean(axis=1), ls, lw=2, label=f"{name} — IoU {iou}")
axes[1].set_title("Courbes précision-rappel (moyenne sur les classes)")
axes[1].set_xlabel("rappel"); axes[1].set_ylabel("précision"); axes[1].legend()
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "metrics_pr.png", bbox_inches="tight"); plt.show()

**Explication :** à gauche, comparaison des métriques principales entre entraînement et validation. À droite, les **courbes précision-rappel** extraites de `COCOeval` (101 points de rappel, moyenne sur les classes présentes) à IoU 0.5 et 0.75 : l'aire sous chaque courbe est exactement l'AP correspondant.

**Interprétation :**

Rappel de lecture : une mAP COCO de 0.37 correspond au modèle de référence entraîné sur les 118 000 images pendant 12 époques ; avec un sous-ensemble et peu d'époques, une valeur nettement plus basse est attendue (voir la comparaison en section 9).

In [ ]:
gap = train_stats[0] - val_stats[0]
s_, m_, l_ = [max(float(x), 0) for x in val_stats[3:6]]
interpretation([
    f"Sur la validation : mAP = {val_stats[0]:.3f}, AP@.5 = {val_stats[1]:.3f}, AP@.75 = {val_stats[2]:.3f}.",
    f"Écart de mAP entre entraînement ({train_stats[0]:.3f}) et validation ({val_stats[0]:.3f}) : {gap:+.3f} → "
    + ("sur-apprentissage marqué : le modèle a en partie mémorisé ses images d'entraînement ; plus de données ou "
       "d'augmentation réduiraient cet écart." if gap > 0.08 else
       "écart modéré : le modèle généralise correctement aux images jamais vues."),
    f"AP small / medium / large = {s_:.3f} / {m_:.3f} / {l_:.3f} : "
    + ("plus l'objet est grand, mieux il est détecté ; les petits objets sont réduits à quelques pixels après les "
       "sous-échantillonnages successifs du ResNet (stride 4 à 32)." if s_ < l_ else
       "fait inhabituel : les petits objets ne sont pas les plus mal détectés sur ce sous-ensemble."),
    f"AR@100 = {val_stats[8]:.3f} : proportion moyenne des objets retrouvés quand on garde 100 détections par image ; "
    f"AR@1 = {val_stats[6]:.3f} montre qu'une seule détection par image ne suffit pas dans des scènes multi-objets.",
    "Courbes précision-rappel : la précision chute quand on cherche à augmenter le rappel ; la courbe à IoU 0.75 "
    "est toujours sous celle à IoU 0.5 car le critère de localisation est plus strict.",
])

### 7.2 Performance par classe

In [ ]:
def per_class_ap(E, coco):
    rows = []
    prec = E.eval["precision"]
    for k, cid in enumerate(E.params.catIds):
        p50 = prec[0, :, k, 0, 2]; p_all = prec[:, :, k, 0, 2]
        n_gt = len(coco.getAnnIds(catIds=cid, imgIds=E.params.imgIds))
        if n_gt == 0:
            continue
        rows.append({"classe": coco.cats[cid]["name"], "objets (GT)": n_gt,
                     "AP@.5": max(p50[p50 > -1].mean(), 0) if (p50 > -1).any() else 0.0,
                     "AP@[.5:.95]": max(p_all[p_all > -1].mean(), 0) if (p_all > -1).any() else 0.0})
    return pd.DataFrame(rows).set_index("classe").sort_values("objets (GT)", ascending=False)

cls_val = per_class_ap(E_val, coco_val)
cls_train = per_class_ap(E_train, coco_train)

top_cls = cls_val.head(15).index
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top_cls))
ax.bar(x - 0.2, cls_train.reindex(top_cls)["AP@.5"].fillna(0), 0.4, label="train", color="#4C72B0")
ax.bar(x + 0.2, cls_val.loc[top_cls, "AP@.5"], 0.4, label="validation", color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels([f"{c}\n(n={cls_val.loc[c, 'objets (GT)']})" for c in top_cls], fontsize=8)
ax.set_ylabel("AP@.5"); ax.set_title("AP@.5 par classe (15 classes les plus fréquentes en validation)"); ax.legend()
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "per_class_ap.png", bbox_inches="tight"); plt.show()
cls_val.head(15).round(3)

**Explication :** pour chaque classe on extrait l'AP depuis le tableau `precision` de `COCOeval` et on le met en regard du nombre d'objets annotés. Le graphe compare l'AP@.5 en entraînement et en validation pour les 15 classes les plus fréquentes en validation.

**Interprétation :**

Points à observer : les classes fréquentes et de grande taille (*person*, véhicules, grands animaux) obtiennent en général les meilleures AP ; les petits objets (*spoon*, *fork*, *traffic light*…) et les classes rares les moins bonnes.

In [ ]:
min_gt = 10 if KAGGLE else 3
c = cls_val[cls_val["objets (GT)"] >= min_gt]
c = c if len(c) >= 6 else cls_val
n_train = pd.Series({coco_train.cats[cid]["name"]: len(coco_train.getAnnIds(catIds=cid)) for cid in coco_train.cats}, name="n")
corr = pd.concat([cls_val["AP@.5"], n_train], axis=1, join="inner").corr(method="spearman").iloc[0, 1]
fmt = lambda s: ", ".join(f"{k} ({v:.2f})" for k, v in s.items())
interpretation([
    f"Classes les mieux détectées (AP@.5, ≥ {min_gt} objets en validation) : {fmt(c['AP@.5'].nlargest(3))}.",
    f"Classes les moins bien détectées : {fmt(c['AP@.5'].nsmallest(3))}.",
    f"Corrélation de Spearman entre le nombre d'exemples d'entraînement d'une classe et son AP@.5 : {corr:.2f} → "
    + ("les classes fréquentes sont nettement mieux apprises : le déséquilibre du dataset se retrouve dans les performances."
       if corr > 0.3 else "lien faible : la difficulté visuelle de la classe (taille, forme, contexte) compte autant que sa fréquence."),
])

### 7.3 Distribution des scores de confiance

In [ ]:
def match_detections(preds, coco, iou_thr=0.5):
    # associe chaque détection (triée par score) à une vérité terrain de même classe -> TP ou FP
    from torchvision.ops import box_iou
    rows = []
    for img_id, p in preds.items():
        gb, gl = gt_boxes(coco, img_id)
        gb, gl = torch.tensor(gb).reshape(-1, 4), torch.tensor(gl)
        used = torch.zeros(len(gl), dtype=torch.bool)
        for box, s, l in zip(p["boxes"], p["scores"], p["labels"]):
            tp = False
            if len(gl):
                ious = box_iou(box[None], gb)[0]
                ious[(gl != l) | used] = 0
                j = int(ious.argmax())
                if ious[j] >= iou_thr:
                    used[j] = tp = True
            rows.append({"score": float(s), "TP": tp})
    return pd.DataFrame(rows)

det = match_detections(val_preds, coco_val)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
bins = np.linspace(0.0, 1, 21)
axes[0].hist([det[det.TP].score, det[~det.TP].score], bins=bins, stacked=True,
             color=["#55A868", "#C44E52"], label=["vrais positifs (IoU ≥ 0.5)", "faux positifs"])
axes[0].set_title("Scores de confiance des détections (validation)"); axes[0].set_xlabel("score"); axes[0].legend()

n_gt = len(coco_val.getAnnIds(imgIds=list(val_preds)))
thr = np.linspace(0.05, 0.95, 19)
prec_t = [det[det.score >= t].TP.mean() if (det.score >= t).any() else np.nan for t in thr]
rec_t = [det[det.score >= t].TP.sum() / n_gt for t in thr]
axes[1].plot(thr, prec_t, "o-", label="précision"); axes[1].plot(thr, rec_t, "s-", label="rappel")
axes[1].axvline(CONFIG["vis_score_thresh"], color="gray", ls=":", label="seuil d'affichage")
axes[1].set_title("Précision / rappel selon le seuil de score (IoU 0.5)"); axes[1].set_xlabel("seuil"); axes[1].legend()
plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "scores.png", bbox_inches="tight"); plt.show()
print(f"{len(det)} détections (score ≥ 0.01) | {det.TP.sum()} vrais positifs | {n_gt} objets annotés")

**Explication :** chaque détection est classée en **vrai positif** (elle recouvre avec IoU ≥ 0.5 un objet de la même classe non encore apparié) ou **faux positif**. L'histogramme montre comment ces deux populations se répartissent selon le score ; la courbe de droite montre le compromis précision/rappel lorsque l'on fait varier le seuil de confiance utilisé en pratique.

**Interprétation :**

Points à observer : un bon détecteur concentre ses vrais positifs vers les scores élevés et ses faux positifs vers les scores faibles ; le seuil de confiance règle le compromis précision / rappel.

In [ ]:
t = CONFIG["vis_score_thresh"]
hi, lo = det[det.score >= t], det[det.score < 0.3]
interpretation([
    f"Au seuil {t} : précision = {hi.TP.mean() if len(hi) else float('nan'):.2f}, rappel = {hi.TP.sum() / max(n_gt, 1):.2f} "
    f"({len(hi)} détections conservées).",
    f"{len(lo) / max(len(det), 1):.0%} des détections ont un score < 0.3 et seulement {lo.TP.mean() if len(lo) else 0:.0%} "
    "d'entre elles sont correctes : les scores faibles correspondent surtout à des faux positifs.",
    "Monter le seuil augmente la précision mais fait baisser le rappel : on choisit un seuil élevé pour l'affichage "
    "(peu de fausses alarmes) et un seuil très bas pour calculer la mAP (qui intègre toute la courbe précision-rappel).",
])

## 8. Exemples de détection d'objets

### 8.1 Images de validation (jamais vues pendant l'entraînement)
Gauche : vérité terrain (vert). Droite : prédictions du modèle avec score ≥ seuil (orange).

In [ ]:
def show_examples(img_ids, coco, img_dir, preds, title, thresh=CONFIG["vis_score_thresh"]):
    fig, axes = plt.subplots(len(img_ids), 2, figsize=(14, 4.6 * len(img_ids)))
    for row, img_id in zip(np.atleast_2d(axes), img_ids):
        img = Image.open(img_dir / coco.imgs[img_id]["file_name"]).convert("RGB")
        gb, gl = gt_boxes(coco, img_id)
        draw_boxes(row[0], img, gb, gl, color="lime")
        row[0].set_title(f"Vérité terrain — image {img_id} ({len(gb)} objets)", fontsize=9)
        p = preds[img_id]
        keep = p["scores"] >= thresh
        draw_boxes(row[1], img, p["boxes"][keep].tolist(), p["labels"][keep].tolist(),
                   p["scores"][keep].tolist(), color="orange")
        row[1].set_title(f"Prédictions (score ≥ {thresh}) — {int(keep.sum())} détections", fontsize=9)
    plt.suptitle(title, fontsize=12, y=1.0); plt.tight_layout(); plt.show()

# on classe les images de validation selon un F1-score par image (au seuil d'affichage)
det_img = []
for img_id, p in val_preds.items():
    sub = match_detections({img_id: {k: v[p["scores"] >= CONFIG["vis_score_thresh"]] for k, v in p.items()}}, coco_val)
    n = len(coco_val.getAnnIds(imgIds=img_id))
    det_img.append({"img": img_id, "TP": sub.TP.sum() if len(sub) else 0, "det": len(sub), "gt": n})
det_img = pd.DataFrame(det_img)
det_img["f1"] = 2 * det_img["TP"] / (det_img["det"] + det_img["gt"]).clip(lower=1)
ranked = det_img.sort_values("f1", ascending=False).img.tolist()
show_examples(ranked[:4], coco_val, VAL_IMG_DIR, val_preds, "Validation — meilleurs cas")

**Explication :** pour choisir des exemples représentatifs, on calcule pour chaque image de validation un **F1-score** (vrais positifs / détections / objets annotés, au seuil d'affichage) et on montre les meilleurs cas, puis ci-dessous les cas les plus difficiles. La colonne de gauche est la vérité terrain, celle de droite la sortie du modèle.

In [ ]:
show_examples(ranked[-3:], coco_val, VAL_IMG_DIR, val_preds, "Validation — cas difficiles", thresh=0.3)

**Explication :** les cas les plus difficiles, affichés avec un seuil abaissé à 0.3 pour voir ce que le modèle « hésite » à détecter.

**Interprétation des exemples de validation :**

- **Meilleurs cas** : objets de taille moyenne ou grande, bien visibles, de classes fréquentes (personnes, véhicules, animaux) : les boîtes prédites recouvrent bien la vérité terrain avec des scores élevés.
- **Cas difficiles** : ils combinent généralement plusieurs difficultés : très petits objets, foules et occlusions, classes rares, objets coupés par le bord de l'image.
- **Erreurs typiques** : confusions entre classes visuellement proches (*car* / *truck*, *cup* / *bowl*, *cat* / *dog*), boîtes en double sur un même objet, boîtes trop larges englobant plusieurs objets, et parfois des objets réellement présents mais **non annotés** dans COCO, comptés à tort comme faux positifs.

### 8.2 Images d'entraînement

In [ ]:
show_examples(sample_ids[:3], coco_train, TRAIN_IMG_DIR, train_preds, "Images d'entraînement — prédictions du modèle")

**Explication :** mêmes visualisations sur des images vues pendant l'entraînement (celles présentées en section 2.4).

**Interprétation :**

Sur les images d'entraînement, les détections sont en général plus complètes et plus confiantes que sur la validation : c'est l'illustration visuelle de l'écart train / validation mesuré en section 7.1. Plus cet écart est grand, plus le modèle a mémorisé ses données au lieu d'apprendre des caractéristiques générales.

## 9. Comparaison avec le modèle de référence (pré-entraîné sur COCO complet)
torchvision fournit le même Faster R-CNN ResNet-50-FPN **entraîné sur les 118 000 images de COCO train2017** (26 époques, plusieurs GPU). L'évaluer sur **nos 500 images de validation** donne la borne de ce que l'architecture peut atteindre et permet de situer notre propre modèle.

In [ ]:
if CONFIG["run_reference"]:
    from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
    ref = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, box_score_thresh=0.01).to(DEVICE)
    ref_raw = predict(ref, val_loader)
    # le modèle de référence prédit directement les identifiants COCO (1..90) -> conversion en étiquettes 1..80
    ref_preds = {}
    for img_id, p in ref_raw.items():
        keep = torch.tensor([int(l) in CAT2LABEL for l in p["labels"]], dtype=torch.bool)
        ref_preds[img_id] = {"boxes": p["boxes"][keep], "scores": p["scores"][keep],
                             "labels": torch.tensor([CAT2LABEL[int(l)] for l in p["labels"][keep]], dtype=torch.int64)}
    print("=== Modèle de référence torchvision ===")
    ref_stats, E_ref = coco_evaluate(coco_val, ref_preds, verbose=True)

    comp_ref = pd.DataFrame({"notre modèle": val_stats, "référence (COCO complet)": ref_stats}, index=METRIC_NAMES).round(4)
    ax = comp_ref.loc[sel].clip(lower=0).plot.bar(figsize=(12, 4.5), rot=30, color=["#DD8452", "#8172B3"])
    ax.set_title("Validation : notre modèle vs modèle de référence torchvision"); ax.set_ylabel("score")
    plt.tight_layout(); plt.savefig(CONFIG["output_dir"] / "reference.png", bbox_inches="tight"); plt.show()
    display(comp_ref)
    interpretation([
        f"Le modèle de référence atteint une mAP de {ref_stats[0]:.3f} sur les mêmes images, contre {val_stats[0]:.3f} pour "
        f"notre modèle (soit {val_stats[0] / max(ref_stats[0], 1e-9):.0%} de la performance de référence).",
        f"Notre modèle a vu {len(train_ds)} images ({len(train_ds) / 118287:.1%} de train2017) pendant {CONFIG['epochs']} époques "
        f"en {512 if KAGGLE else 448}px ; la référence a vu les 118 287 images pendant 26 époques en 800px.",
        "L'architecture est identique : l'écart s'explique entièrement par la quantité de données et le budget d'entraînement, "
        "ce qui illustre le besoin en données des détecteurs profonds.",
    ])
    show_examples(ranked[:2], coco_val, VAL_IMG_DIR, ref_preds, "Modèle de référence — mêmes images que la section 8.1")

**Explication :** on charge les poids officiels `COCO_V1` de torchvision, on prédit sur exactement les mêmes images de validation, on convertit les identifiants de catégories COCO (1…90) vers nos étiquettes (1…80) puis on calcule les mêmes métriques. Le graphe et l'interprétation situent notre modèle par rapport à la performance de l'architecture entraînée sur tout COCO ; les exemples permettent de comparer visuellement avec la section 8.1.

## 10. Conclusion et pistes d'amélioration

**Bilan.** Ce TP a couvert toute la chaîne d'un détecteur d'objets moderne :
1. **Données** : chargement de COCO 2017 au format officiel, construction de sous-ensembles reproductibles, analyse du déséquilibre des classes et de la taille des objets ;
2. **Modèle** : Faster R-CNN à deux étapes avec un backbone **ResNet-50 pré-entraîné sur ImageNet** et un **FPN**, un **RPN** pour proposer des régions, **RoI Align** et une tête de classification / régression à 81 sorties ;
3. **Entraînement** : SGD avec warm-up et décroissance cosinus, précision mixte, gradient clipping, sauvegarde du meilleur modèle ;
4. **Évaluation** : métriques officielles COCO (mAP@[.5:.95], AP@.5, AP@.75, AP par taille, AR), courbes précision-rappel, AP par classe, analyse des scores et exemples visuels.

Les blocs « Interprétation » ci-dessus donnent les valeurs obtenues. On retrouve les tendances classiques de la détection sur COCO : les **petits objets** et les **classes rares** sont les plus difficiles, la **localisation stricte** (IoU 0.75) est bien plus exigeante que la simple détection (IoU 0.5), et un sous-ensemble limité de données provoque un **écart train / validation**.

**Pistes d'amélioration**
- **Plus de données et d'époques** : c'est le facteur principal (cf. section 9) ; passer à 20 000 images ou à COCO complet ;
- **Résolution d'entrée plus grande** (800×1333) pour mieux détecter les petits objets ;
- **Augmentations plus riches** : *multi-scale training*, recadrages aléatoires, variations de couleur, *mosaic* ;
- **Fine-tuning** à partir des poids COCO pré-entraînés lorsque l'objectif est la performance plutôt que l'apprentissage complet ;
- **Architectures plus récentes** : Faster R-CNN v2 (ResNet-50-FPN v2), Cascade R-CNN, ou détecteurs à une étape (RetinaNet, YOLO) et à base de *transformers* (DETR) selon le compromis vitesse / précision recherché.

In [ ]:
# Récapitulatif final (sauvegardé dans outputs/)
recap = {
    "mode": MODE, "mAP_reference_torchvision": round(float(ref_stats[0]), 4) if CONFIG["run_reference"] else None,
    "images_train": len(train_ds), "images_val": len(val_ds), "epochs": CONFIG["epochs"],
    "meilleure_epoque": int(epoch_df.val_mAP.idxmax()),
    **{f"val_{k}": round(float(v), 4) for k, v in zip(METRIC_NAMES, val_stats)},
    **{f"train_{k}": round(float(v), 4) for k, v in zip(METRIC_NAMES[:3], train_stats[:3])},
    "inference_ms_par_image": round(1000 / fps, 1),
}
json.dump(recap, open(CONFIG["output_dir"] / "recap.json", "w"), indent=2)
pd.Series(recap, name="valeur").to_frame()

**Explication :** ce dernier bloc rassemble les résultats clés et les enregistre dans `outputs/recap.json` (avec le modèle `faster_rcnn_r50_fpn.pth`, l'historique `history.json` et les figures `.png`), pour pouvoir les reprendre dans un rapport. Sur Kaggle, ces fichiers sont dans l'onglet **Output** du notebook (téléchargeables après « Save Version »).